In [ ]:
import pandas as pd

csv_path = "/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/birds_embeddings_birdnet_padded_3s.csv"
df = pd.read_csv(csv_path)
df

Dimentionality reduction via UMAP - 30 components

In [2]:
import pandas as pd
import numpy as np
import umap
import matplotlib.pyplot as plt

# Load the CSV with embeddings and calltype
csv_path = "/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/birds_embeddings_birdnet_padded_3s.csv"
df = pd.read_csv(csv_path)

print(f"Loaded data with shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:5]}...")  # Show first 5 columns

# Separate metadata from embeddings
# Assuming 'calltype' and 'file' are metadata columns, rest are embeddings
metadata_cols = ['calltype', 'file']
embedding_cols = [col for col in df.columns if col not in metadata_cols]

print(f"\nNumber of embedding features: {len(embedding_cols)}")

# Extract embeddings as numpy array
X = df[embedding_cols].values
y = df['calltype'].values

# Run UMAP
print("\nRunning UMAP...")
reducer = umap.UMAP(n_components=30, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = reducer.fit_transform(X)

print("UMAP transformation complete!")

# Create a DataFrame with UMAP results
umap_df = pd.DataFrame({
    'UMAP1': X_umap[:, 0],
    'UMAP2': X_umap[:, 1],
    'calltype': y
})

2026-01-26 18:42:26.890005: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loaded data with shape: (6047, 1028)
Columns: ['calltype', 'file', 'start_time', 'end_time', '0']...

Number of embedding features: 1026

Running UMAP...


/home/Shelby/miniconda3/envs/BlackbirdOP_py312/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP transformation complete!


In [ ]:
# HDBSCAN provides membership probabilities and outlier scores
df['cluster_probability'] = clusterer.probabilities_
df['outlier_score'] = clusterer.outlier_scores_

print("Cluster membership probabilities:")
print(df[['calltype', 'cluster', 'cluster_probability', 'outlier_score']].head(10))

# Visualize with probability-based transparency
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    X_umap[:, 0], 
    X_umap[:, 1], 
    c=cluster_labels,
    cmap='Spectral',
    s=10,
    alpha=clusterer.probabilities_  # Alpha based on membership probability
)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('HDBSCAN Clusters (Transparency = Membership Probability)')
plt.tight_layout()
plt.show()

### Cluster Probabilities and Outlier Scores

In [ ]:
# Cross-tabulation of clusters vs call types
print("Cross-tabulation of Clusters vs Call Types:")
crosstab = pd.crosstab(df['cluster'], df['calltype'], margins=True)
print(crosstab)

# Calculate purity metrics
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, homogeneity_completeness_v_measure

# Filter out noise points for some metrics
non_noise_mask = cluster_labels != -1
if sum(non_noise_mask) > 0:
    ari = adjusted_rand_score(y[non_noise_mask], cluster_labels[non_noise_mask])
    nmi = normalized_mutual_info_score(y[non_noise_mask], cluster_labels[non_noise_mask])
    homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y[non_noise_mask], cluster_labels[non_noise_mask])
    
    print(f"\n--- Clustering Quality Metrics (excluding noise) ---")
    print(f"Adjusted Rand Index: {ari:.3f}")
    print(f"Normalized Mutual Information: {nmi:.3f}")
    print(f"Homogeneity: {homogeneity:.3f}")
    print(f"Completeness: {completeness:.3f}")
    print(f"V-measure: {v_measure:.3f}")

### Compare Clusters with Ground Truth Call Types

In [ ]:
# Visualize clusters in 2D UMAP space
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    X_umap[:, 0], 
    X_umap[:, 1], 
    c=cluster_labels, 
    cmap='Spectral',
    s=10,
    alpha=0.7
)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('HDBSCAN Clusters on UMAP-reduced Embeddings')
plt.tight_layout()
plt.show()

# Noise points in gray
plt.figure(figsize=(12, 8))
colors = ['gray' if label == -1 else f'C{label % 10}' for label in cluster_labels]
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=colors, s=10, alpha=0.7)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('HDBSCAN Clusters (Gray = Noise)')
plt.tight_layout()
plt.show()

### Visualize Clusters (2D UMAP)

In [ ]:
import hdbscan

# Run HDBSCAN on the UMAP-reduced embeddings
print("Running HDBSCAN clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,      # Minimum samples in a cluster
    min_samples=3,            # Minimum samples in neighborhood
    metric='euclidean',
    cluster_selection_epsilon=0.0,
    cluster_selection_method='eom'
)

# Fit the clusterer
cluster_labels = clusterer.fit_predict(X_umap)

# Add cluster labels to the dataframe
df['cluster'] = cluster_labels

print(f"Clustering complete!")
print(f"Number of clusters found: {len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)}")
print(f"Number of noise points: {sum(cluster_labels == -1)}")
print(f"Cluster distribution:\n{pd.Series(cluster_labels).value_counts().sort_index()}")

## HDBSCAN Clustering